Step-by-step implementation of the location-allocation methodology in:

    Zhu, Chen, Zhang, and Qiao (2020),
    "Optimal fuel station arrangement for multiple GEO spacecraft refueling mission",
    Advances in Space Research 66(8), 1924-1936.
    DOI: 10.1016/j.asr.2020.07.028

The paper's high-level workflow is reproduced here as:

1. Load the GEO target catalog and orbital-element history.
2. Select the latest public orbital state for every target.
3. Propagate every target to one common analysis epoch.
4. Convert the states to GEO longitudes (the continuous location space).
5. Attach target fuel demands and define servicing-spacecraft capacity.
6. Generate station-to-target maneuver matrices inside the optimizer.
7. Use capacity-aware greedy allocation for each proposed station layout.
8. Use a gravitational search algorithm (GSA) to move the station locations.
9. Find the smallest feasible station count first.
10. Re-optimize that station count for minimum total servicing cost.
11. Export station locations, assignments, trade-study results, and plots.

IMPORTANT DATA LIMITATION
-------------------------
The supplied catalog does not contain target fuel demand q_i or SSC capacity C.
This script therefore runs in DEMO_MODE with clearly labeled placeholder values.
Replace those values (or provide a demand CSV) before treating results as a
mission-design conclusion.

The code uses a transparent coplanar GEO phasing model for the maneuver-cost
oracle. If you have the exact transfer-cost routine used in your research, only
`sortie_delta_v_matrix` needs to be replaced; the location-allocation solver
does not change.

Run:
    python geo_fuel_station_methodology.py

Required packages:
    numpy, pandas, matplotlib

In [26]:
# %% STEP 0 — Imports and user configuration

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [27]:
# -------------------------------
# Input and output configuration
# -------------------------------

STATUS_FILENAME = "us_geo_catalog_status.csv"
ACTIVE_CATALOG_FILENAME = "celestrak_active_gpz_catalog.csv"
HISTORY_FILENAME = "gp_history_20260801_20260901_3cbd5060be.json"

# Leave as None to let the script search ./upload, the script directory, and cwd.
DATA_DIRECTORY = Path("data/geo_us")
OUTPUT_DIRECTORY = Path("geo_fuel_station_outputs")

# Leave as None to use the latest epoch found in the supplied history.
ANALYSIS_EPOCH_UTC: str | None = None

In [28]:
# ------------------------------------------
# Mission assumptions that YOU must replace
# ------------------------------------------

DEMO_MODE = True

# Option A: provide a CSV containing NORAD_CAT_ID and DEMAND_KG.
DEMAND_CSV: Path | None = None
DEMAND_ID_COLUMN = "NORAD_CAT_ID"
DEMAND_VALUE_COLUMN = "DEMAND_KG"

# Option B: demonstration-only uniform demand used when DEMAND_CSV is None.
UNIFORM_TARGET_DEMAND_KG = 100.0

# Capacity available to the reusable SSC associated with each station.
# The capacity constraint is sum(q_i assigned to station j) <= C.
# Total propellant stored at each fuel station over the planning horizon.
STATION_INVENTORY_KG = 2_500.0

# Propellant the SSC can carry and deliver on one station → target → station sortie.
SSC_PAYLOAD_CAPACITY_KG = 200.0

# If not None, a target is accessible only when its round-trip sortie delta-v
# is at or below this limit. Set this from the SSC fuel budget.
MAX_SORTIE_DV_KM_S: float | None = 0.70

# Circular GEO phasing model. One phasing revolution is the fastest/highest-dv
# option. More revolutions reduce delta-v but increase transfer time.
PHASING_REVOLUTIONS = 1

# The solver minimizes either total round-trip delta-v or a rocket-equation
# propellant estimate. "propellant_kg" requires dry mass and Isp assumptions.
OBJECTIVE_MODE = "delta_v"  # "delta_v" or "propellant_kg"
SSC_DRY_MASS_KG = 1_000.0
SSC_ISP_SECONDS = 320.0

In [29]:
# ------------------------
# Optimization configuration
# ------------------------

RANDOM_SEED = 27
MAX_STATIONS_TO_TEST = 10

# Fast first pass used to identify the smallest feasible station count.
SCREEN_POPULATION = 16
SCREEN_ITERATIONS = 40
SCREEN_RESTARTS = 1

# Longer second pass used to minimize cost for the selected station count.
FINAL_POPULATION = 24
FINAL_ITERATIONS = 80
FINAL_RESTARTS = 2

# Greedy allocation is repeated with slightly different target orders; the best
# capacity-feasible allocation is kept.
GREEDY_ALLOCATION_ATTEMPTS = 6

# These defaults are sized for an interactive first run. For a publication
# trade study, increase them (for example 40/200/5 and 60/500/10) and confirm
# that independent seeds converge to the same layouts and objective values.

# GSA constants.
GSA_G0 = 100.0
GSA_ALPHA = 20.0

# Large lexicographic penalties: feasibility dominates cost.
UNASSIGNED_TARGET_PENALTY = 1_000_000.0
CAPACITY_EXCESS_PENALTY = 100_000.0

In [30]:
# %% STEP 1 — Locate and load the three supplied files


def locate_input(filename: str) -> Path:
    """Find an input file without hard-coding the temporary workspace path."""
    candidates: list[Path] = []
    if DATA_DIRECTORY is not None:
        candidates.append(Path(DATA_DIRECTORY) / filename)

    script_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates.extend(
        [
            Path.cwd() / "upload" / filename,
            script_dir / "upload" / filename,
            Path.cwd() / filename,
            script_dir / filename,
        ]
    )
    for path in candidates:
        if path.exists():
            return path.resolve()

    searched = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    status_path = locate_input(STATUS_FILENAME)
    active_path = locate_input(ACTIVE_CATALOG_FILENAME)
    history_path = locate_input(HISTORY_FILENAME)

    status = pd.read_csv(status_path)
    active = pd.read_csv(active_path)
    with history_path.open("r", encoding="utf-8") as handle:
        history = pd.DataFrame(json.load(handle))

    status["NORAD_CAT_ID"] = status["NORAD_CAT_ID"].astype(str)
    active["NORAD_CAT_ID"] = active["NORAD_CAT_ID"].astype(str)
    history["NORAD_CAT_ID"] = history["NORAD_CAT_ID"].astype(str)
    history["EPOCH"] = pd.to_datetime(history["EPOCH"], utc=True)

    print(f"Status targets: {len(status):,}")
    print(f"Active GPZ catalog objects: {len(active):,}")
    print(f"History records: {len(history):,}")
    print(f"History satellites: {history['NORAD_CAT_ID'].nunique():,}")
    return status, active, history



In [31]:
# %% STEP 2 — Keep the latest public orbital state for every catalog target


ORBITAL_NUMERIC_COLUMNS = [
    "MEAN_MOTION",
    "ECCENTRICITY",
    "INCLINATION",
    "RA_OF_ASC_NODE",
    "ARG_OF_PERICENTER",
    "MEAN_ANOMALY",
    "SEMIMAJOR_AXIS",
]


def latest_states_for_catalog(
    status: pd.DataFrame, active: pd.DataFrame, history: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    status = status.copy()
    active_ids = set(active["NORAD_CAT_ID"])
    status["IN_ACTIVE_GPZ_CATALOG"] = status["NORAD_CAT_ID"].isin(active_ids)
    print(
        "Status targets also present in active GPZ catalog: "
        f"{status['IN_ACTIVE_GPZ_CATALOG'].sum():,}/{len(status):,}"
    )

    catalog_ids = set(status["NORAD_CAT_ID"])
    matching = history[history["NORAD_CAT_ID"].isin(catalog_ids)].copy()

    for column in ORBITAL_NUMERIC_COLUMNS:
        matching[column] = pd.to_numeric(matching[column], errors="coerce")

    required = [
        "EPOCH",
        "MEAN_MOTION",
        "ECCENTRICITY",
        "INCLINATION",
        "RA_OF_ASC_NODE",
        "ARG_OF_PERICENTER",
        "MEAN_ANOMALY",
        "SEMIMAJOR_AXIS",
    ]
    matching = matching.dropna(subset=required)

    latest = (
        matching.sort_values("EPOCH")
        .groupby("NORAD_CAT_ID", as_index=False)
        .tail(1)
        .copy()
    )

    # Add current operating status without overwriting the OMM orbital fields.
    catalog_metadata = status[
        ["NORAD_CAT_ID", "OPS_STATUS_CODE", "IN_ACTIVE_GPZ_CATALOG"]
    ].drop_duplicates("NORAD_CAT_ID")
    latest = latest.merge(catalog_metadata, on="NORAD_CAT_ID", how="left")

    included_ids = set(latest["NORAD_CAT_ID"])
    excluded = status[~status["NORAD_CAT_ID"].isin(included_ids)].copy()
    excluded["EXCLUSION_REASON"] = "No complete public GP/OMM state in history file"

    print(f"Usable targets: {len(latest):,}")
    print(f"Excluded targets: {len(excluded):,}")
    return latest.reset_index(drop=True), excluded.reset_index(drop=True)


In [32]:
# %% STEP 3 — Propagate mean elements to one common epoch


MU_EARTH_KM3_S2 = 398_600.4418
GEO_RADIUS_KM = 42_164.0
SIDEREAL_DAY_S = 86_164.0905
STANDARD_GRAVITY_M_S2 = 9.80665


def wrap_to_pi(angle_rad: np.ndarray | float) -> np.ndarray:
    """Wrap radians to [-pi, pi)."""
    return (np.asarray(angle_rad) + np.pi) % (2.0 * np.pi) - np.pi


def solve_kepler(mean_anomaly_rad: np.ndarray, eccentricity: np.ndarray) -> np.ndarray:
    """Vectorized Newton solve for E - e sin(E) = M."""
    mean_anomaly_rad = np.mod(mean_anomaly_rad, 2.0 * np.pi)
    eccentric_anomaly = mean_anomaly_rad.copy()
    for _ in range(20):
        residual = eccentric_anomaly - eccentricity * np.sin(eccentric_anomaly) - mean_anomaly_rad
        derivative = 1.0 - eccentricity * np.cos(eccentric_anomaly)
        step = residual / derivative
        eccentric_anomaly -= step
        if np.max(np.abs(step)) < 1e-13:
            break
    return eccentric_anomaly


def julian_date(timestamp: pd.Timestamp) -> float:
    """UTC timestamp to Julian date; adequate for the GMST approximation below."""
    return timestamp.timestamp() / 86_400.0 + 2_440_587.5


def greenwich_mean_sidereal_time(timestamp: pd.Timestamp) -> float:
    """Return GMST in radians using the standard low-order expression."""
    jd = julian_date(timestamp)
    centuries = (jd - 2_451_545.0) / 36_525.0
    gmst_deg = (
        280.46061837
        + 360.98564736629 * (jd - 2_451_545.0)
        + 0.000387933 * centuries**2
        - centuries**3 / 38_710_000.0
    )
    return np.deg2rad(gmst_deg % 360.0)


def propagate_mean_elements_to_eci(
    states: pd.DataFrame, analysis_epoch: pd.Timestamp
) -> np.ndarray:
    """
    Propagate two-body mean elements and return ECI position vectors in km.

    TLE/OMM elements are SGP4 mean elements, so this is a transparent research
    prototype rather than a replacement for an operational SGP4 propagator.
    Every target is nevertheless compared at exactly the same epoch.
    """
    dt_seconds = (analysis_epoch - states["EPOCH"]).dt.total_seconds().to_numpy()
    mean_motion_rad_s = states["MEAN_MOTION"].to_numpy() * 2.0 * np.pi / 86_400.0
    mean_anomaly = np.deg2rad(states["MEAN_ANOMALY"].to_numpy()) + mean_motion_rad_s * dt_seconds

    eccentricity = states["ECCENTRICITY"].to_numpy()
    eccentric_anomaly = solve_kepler(mean_anomaly, eccentricity)
    true_anomaly = 2.0 * np.arctan2(
        np.sqrt(1.0 + eccentricity) * np.sin(eccentric_anomaly / 2.0),
        np.sqrt(1.0 - eccentricity) * np.cos(eccentric_anomaly / 2.0),
    )

    semimajor_axis = states["SEMIMAJOR_AXIS"].to_numpy()
    radius = semimajor_axis * (1.0 - eccentricity * np.cos(eccentric_anomaly))
    arg_latitude = np.deg2rad(states["ARG_OF_PERICENTER"].to_numpy()) + true_anomaly
    inclination = np.deg2rad(states["INCLINATION"].to_numpy())
    raan = np.deg2rad(states["RA_OF_ASC_NODE"].to_numpy())

    cos_u, sin_u = np.cos(arg_latitude), np.sin(arg_latitude)
    cos_o, sin_o = np.cos(raan), np.sin(raan)
    cos_i, sin_i = np.cos(inclination), np.sin(inclination)

    x = radius * (cos_o * cos_u - sin_o * sin_u * cos_i)
    y = radius * (sin_o * cos_u + cos_o * sin_u * cos_i)
    z = radius * (sin_u * sin_i)
    return np.column_stack([x, y, z])


def eci_to_ecef(eci_km: np.ndarray, analysis_epoch: pd.Timestamp) -> np.ndarray:
    theta = greenwich_mean_sidereal_time(analysis_epoch)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    x = cos_t * eci_km[:, 0] + sin_t * eci_km[:, 1]
    y = -sin_t * eci_km[:, 0] + cos_t * eci_km[:, 1]
    return np.column_stack([x, y, eci_km[:, 2]])


def prepare_target_locations(states: pd.DataFrame) -> tuple[pd.DataFrame, pd.Timestamp]:
    if ANALYSIS_EPOCH_UTC is None:
        analysis_epoch = states["EPOCH"].max()
    else:
        analysis_epoch = pd.Timestamp(ANALYSIS_EPOCH_UTC)
        if analysis_epoch.tzinfo is None:
            analysis_epoch = analysis_epoch.tz_localize("UTC")
        else:
            analysis_epoch = analysis_epoch.tz_convert("UTC")

    eci = propagate_mean_elements_to_eci(states, analysis_epoch)
    ecef = eci_to_ecef(eci, analysis_epoch)

    result = states.copy()
    result["ANALYSIS_EPOCH_UTC"] = analysis_epoch.isoformat()
    result["ECI_X_KM"] = eci[:, 0]
    result["ECI_Y_KM"] = eci[:, 1]
    result["ECI_Z_KM"] = eci[:, 2]
    result["ECEF_X_KM"] = ecef[:, 0]
    result["ECEF_Y_KM"] = ecef[:, 1]
    result["ECEF_Z_KM"] = ecef[:, 2]
    result["LONGITUDE_RAD"] = np.arctan2(ecef[:, 1], ecef[:, 0])
    result["LONGITUDE_DEG"] = np.rad2deg(result["LONGITUDE_RAD"])
    result["GEOCENTRIC_LATITUDE_DEG"] = np.rad2deg(
        np.arctan2(ecef[:, 2], np.hypot(ecef[:, 0], ecef[:, 1]))
    )

    print(f"Common analysis epoch: {analysis_epoch.isoformat()}")
    return result, analysis_epoch


In [33]:
# %% STEP 4 — Attach each target's refueling demand q_i


def attach_demands(targets: pd.DataFrame) -> pd.DataFrame:
    result = targets.copy()

    if DEMAND_CSV is not None:
        demand_path = Path(DEMAND_CSV)
        demand = pd.read_csv(demand_path)
        demand[DEMAND_ID_COLUMN] = demand[DEMAND_ID_COLUMN].astype(str)
        demand[DEMAND_VALUE_COLUMN] = pd.to_numeric(
            demand[DEMAND_VALUE_COLUMN], errors="raise"
        )
        result = result.merge(
            demand[[DEMAND_ID_COLUMN, DEMAND_VALUE_COLUMN]],
            left_on="NORAD_CAT_ID",
            right_on=DEMAND_ID_COLUMN,
            how="left",
        )
        result["DEMAND_KG"] = result[DEMAND_VALUE_COLUMN]
        if result["DEMAND_KG"].isna().any():
            missing = result.loc[result["DEMAND_KG"].isna(), "NORAD_CAT_ID"].tolist()
            raise ValueError(f"Demand file is missing NORAD IDs: {missing}")
    else:
        result["DEMAND_KG"] = float(UNIFORM_TARGET_DEMAND_KG)
        if DEMO_MODE:
            warnings.warn(
                "DEMO_MODE: using a uniform placeholder demand for every target. "
                "Replace this before reporting mission-design results.",
                stacklevel=2,
            )

    if (result["DEMAND_KG"] <= 0).any():
        raise ValueError("Every target demand must be positive.")
    if (result["DEMAND_KG"] > SSC_PAYLOAD_CAPACITY_KG).any():
        bad = result.loc[
            result["DEMAND_KG"] > SSC_PAYLOAD_CAPACITY_KG,
            ["NORAD_CAT_ID", "OBJECT_NAME", "DEMAND_KG"],
        ]
        raise ValueError(
            "At least one target requires more propellant than the SSC can carry "
            "on one sortie:\n" + bad.to_string(index=False)
        )
    return result


In [34]:
import kmapper as km
from sklearn.cluster import DBSCAN

mapper = km.KeplerMapper(verbose=1)

cover = km.Cover(
    n_cubes=10,
    perc_overlap=0.30
)

graph = mapper.map(
    lens,
    X=X,
    cover=cover,
    clusterer=DBSCAN(
        eps=3000,
        min_samples=2
    )
)

print("Mapper nodes:", len(graph["nodes"]))
print("Mapper links:", sum(len(v) for v in graph["links"].values()))

from pathlib import Path
from IPython.display import HTML, display

html_path = Path("geo_mapper.html")

mapper.visualize(
    graph,
    path_html=str(html_path),
    title=f"US GEO Mapper Graph — {t}",
    custom_tooltips=snapshot["object_name"].astype(str).values
)

import webbrowser

webbrowser.open(
    Path("geo_mapper.html").resolve().as_uri()
)

KeplerMapper(verbose=1)


NameError: name 'lens' is not defined

In [ ]:
# %% STEP 5 — Define the station-to-target maneuver-cost oracle


def phasing_one_way_delta_v_km_s(
    angular_separation_rad: np.ndarray,
    revolutions: int = PHASING_REVOLUTIONS,
) -> np.ndarray:
    """
    Two-impulse, coplanar GEO phasing delta-v from a station to a target.

    The code evaluates the shorter/faster inner phasing orbit and the longer/
    slower outer phasing orbit and keeps the less expensive valid option.
    The returned value already includes insertion into and exit from the phasing
    orbit, but not the return trip to the fuel station.
    """
    if revolutions < 1:
        raise ValueError("PHASING_REVOLUTIONS must be at least 1.")

    separation = np.abs(wrap_to_pi(angular_separation_rad))
    geo_period = 2.0 * np.pi * np.sqrt(GEO_RADIUS_KM**3 / MU_EARTH_KM3_S2)
    geo_speed = np.sqrt(MU_EARTH_KM3_S2 / GEO_RADIUS_KM)

    period_inner = geo_period * (1.0 - separation / (2.0 * np.pi * revolutions))
    period_outer = geo_period * (1.0 + separation / (2.0 * np.pi * revolutions))

    def two_impulse_cost(period: np.ndarray) -> np.ndarray:
        semimajor_axis = (
            MU_EARTH_KM3_S2 * (period / (2.0 * np.pi)) ** 2
        ) ** (1.0 / 3.0)
        speed_at_geo = np.sqrt(
            MU_EARTH_KM3_S2 * (2.0 / GEO_RADIUS_KM - 1.0 / semimajor_axis)
        )
        return 2.0 * np.abs(speed_at_geo - geo_speed)

    return np.minimum(two_impulse_cost(period_inner), two_impulse_cost(period_outer))


def sortie_delta_v_matrix(
    station_longitudes_rad: np.ndarray,
    target_longitudes_rad: np.ndarray,
) -> np.ndarray:
    """
    Build a target x station cost matrix for proposed station locations.

    This function is called *inside* the optimizer. Station locations therefore
    do not need to be known before optimization.
    """
    separation = target_longitudes_rad[:, None] - station_longitudes_rad[None, :]
    one_way = phasing_one_way_delta_v_km_s(separation)
    return 2.0 * one_way  # station -> target -> station


def sortie_propellant_matrix(
    sortie_delta_v_km_s: np.ndarray,
    target_demands_kg: np.ndarray,
) -> np.ndarray:
    """
    Approximate maneuver propellant for one independent round-trip sortie.

    The outbound leg carries q_i; the return leg is unloaded. Delivered target
    propellant is not counted as maneuver propellant. This is an optional cost
    model, not a substitute for a full mass/trajectory simulation.
    """
    one_way_m_s = 0.5 * sortie_delta_v_km_s * 1_000.0
    exhaust_velocity = SSC_ISP_SECONDS * STANDARD_GRAVITY_M_S2
    ratio = np.exp(one_way_m_s / exhaust_velocity)

    # Work backward from dry SSC after the return burn.
    before_return_burn = SSC_DRY_MASS_KG * ratio
    before_outbound_burn = (before_return_burn + target_demands_kg[:, None]) * ratio
    return before_outbound_burn - (SSC_DRY_MASS_KG + target_demands_kg[:, None])


def objective_cost_matrix(
    station_longitudes_rad: np.ndarray,
    target_longitudes_rad: np.ndarray,
    target_demands_kg: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    sortie_dv = sortie_delta_v_matrix(station_longitudes_rad, target_longitudes_rad)
    if OBJECTIVE_MODE == "delta_v":
        objective = sortie_dv
    elif OBJECTIVE_MODE == "propellant_kg":
        objective = sortie_propellant_matrix(sortie_dv, target_demands_kg)
    else:
        raise ValueError("OBJECTIVE_MODE must be 'delta_v' or 'propellant_kg'.")
    return objective, sortie_dv


In [ ]:
# %% STEP 6 — Capacity-aware greedy target allocation


@dataclass
class AllocationResult:
    assignment: np.ndarray
    station_load_kg: np.ndarray
    total_objective: float
    total_sortie_dv_km_s: float
    unassigned_count: int
    assigned_cost: np.ndarray
    assigned_sortie_dv_km_s: np.ndarray


def one_greedy_allocation(
    objective_matrix: np.ndarray,
    sortie_dv_matrix: np.ndarray,
    demands_kg: np.ndarray,
    target_order: np.ndarray,
) -> AllocationResult:
    n_targets, n_stations = objective_matrix.shape
    assignment = np.full(n_targets, -1, dtype=int)
    remaining = np.full(n_stations, STATION_INVENTORY_KG, dtype=float)

    for target in target_order:
        feasible_capacity = remaining + 1e-10 >= demands_kg[target]
        if MAX_SORTIE_DV_KM_S is None:
            feasible_access = np.ones(n_stations, dtype=bool)
        else:
            feasible_access = sortie_dv_matrix[target] <= MAX_SORTIE_DV_KM_S + 1e-12

        candidates = np.flatnonzero(feasible_capacity & feasible_access)
        if candidates.size == 0:
            continue

        station = candidates[np.argmin(objective_matrix[target, candidates])]
        assignment[target] = station
        remaining[station] -= demands_kg[target]

    assigned_mask = assignment >= 0
    assigned_cost = np.full(n_targets, np.nan)
    assigned_sortie_dv = np.full(n_targets, np.nan)
    target_indices = np.flatnonzero(assigned_mask)
    assigned_cost[target_indices] = objective_matrix[
        target_indices, assignment[target_indices]
    ]
    assigned_sortie_dv[target_indices] = sortie_dv_matrix[
        target_indices, assignment[target_indices]
    ]

    station_load = np.zeros(n_stations)
    for station in range(n_stations):
        station_load[station] = demands_kg[assignment == station].sum()

    return AllocationResult(
        assignment=assignment,
        station_load_kg=station_load,
        total_objective=float(np.nansum(assigned_cost)),
        total_sortie_dv_km_s=float(np.nansum(assigned_sortie_dv)),
        unassigned_count=int((~assigned_mask).sum()),
        assigned_cost=assigned_cost,
        assigned_sortie_dv_km_s=assigned_sortie_dv,
    )


def greedy_capacity_allocation(
    objective_matrix: np.ndarray,
    sortie_dv_matrix: np.ndarray,
    demands_kg: np.ndarray,
    rng: np.random.Generator,
    attempts: int = GREEDY_ALLOCATION_ATTEMPTS,
) -> AllocationResult:
    """Repeat the greedy allocator with demand/regret-aware target orders."""
    if objective_matrix.shape != sortie_dv_matrix.shape:
        raise ValueError("Objective and delta-v matrices must have the same shape.")

    sorted_cost = np.sort(objective_matrix, axis=1)
    regret = (
        sorted_cost[:, 1] - sorted_cost[:, 0]
        if objective_matrix.shape[1] > 1
        else np.zeros(len(demands_kg))
    )

    orders = [
        np.lexsort((-regret, -demands_kg)),
        np.argsort(-demands_kg),
        np.argsort(-regret),
    ]
    while len(orders) < max(3, attempts):
        jitter = rng.normal(0.0, 0.05, len(demands_kg))
        priority = demands_kg / demands_kg.max() + regret / (regret.max() + 1e-12) + jitter
        orders.append(np.argsort(-priority))

    best: AllocationResult | None = None
    for order in orders[:attempts]:
        candidate = one_greedy_allocation(
            objective_matrix, sortie_dv_matrix, demands_kg, order
        )
        key = (candidate.unassigned_count, candidate.total_objective)
        if best is None or key < (best.unassigned_count, best.total_objective):
            best = candidate

    assert best is not None
    return best


In [ ]:
# %% STEP 7 — Score any continuous set of proposed fuel-station locations


@dataclass
class LayoutEvaluation:
    station_longitudes_rad: np.ndarray
    fitness: float
    allocation: AllocationResult


def evaluate_layout(
    station_longitudes_rad: np.ndarray,
    target_longitudes_rad: np.ndarray,
    demands_kg: np.ndarray,
    rng: np.random.Generator,
) -> LayoutEvaluation:
    stations = wrap_to_pi(station_longitudes_rad)
    objective_matrix, sortie_dv = objective_cost_matrix(
        stations, target_longitudes_rad, demands_kg
    )
    allocation = greedy_capacity_allocation(
        objective_matrix, sortie_dv, demands_kg, rng
    )

    capacity_excess = np.maximum(allocation.station_load_kg - STATION_INVENTORY_KG, 0.0).sum()
    fitness = (
        allocation.total_objective
        + UNASSIGNED_TARGET_PENALTY * allocation.unassigned_count
        + CAPACITY_EXCESS_PENALTY * capacity_excess
    )
    return LayoutEvaluation(stations, float(fitness), allocation)


In [ ]:
# %% STEP 8 — Gravitational Search Algorithm (the station-location update)


def circular_weighted_center(angles: np.ndarray, weights: np.ndarray) -> float:
    vector = np.sum(weights * np.exp(1j * angles))
    if abs(vector) < 1e-14:
        return float(angles[0])
    return float(np.angle(vector))


def seeded_population(
    n_stations: int,
    population_size: int,
    target_longitudes_rad: np.ndarray,
    demands_kg: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    """Mix data-informed and uniform-random initial station layouts."""
    population = rng.uniform(-np.pi, np.pi, size=(population_size, n_stations))

    # One deterministic circular-quantile layout, weighted by demand.
    order = np.argsort(target_longitudes_rad)
    sorted_lon = target_longitudes_rad[order]
    sorted_w = demands_kg[order]
    cumulative = np.cumsum(sorted_w) / sorted_w.sum()
    quantiles = (np.arange(n_stations) + 0.5) / n_stations
    population[0] = np.interp(quantiles, cumulative, sorted_lon)

    # Additional layouts sample actual target longitudes, then perturb them.
    probabilities = demands_kg / demands_kg.sum()
    for agent in range(1, min(population_size, 1 + population_size // 3)):
        replace = n_stations > len(target_longitudes_rad)
        centers = rng.choice(
            target_longitudes_rad,
            size=n_stations,
            replace=replace,
            p=probabilities,
        )
        population[agent] = wrap_to_pi(centers + rng.normal(0.0, 0.12, n_stations))
    return population


def gravitational_search(
    n_stations: int,
    target_longitudes_rad: np.ndarray,
    demands_kg: np.ndarray,
    population_size: int,
    iterations: int,
    seed: int,
) -> LayoutEvaluation:
    """Continuous circular GSA search over k station longitude variables."""
    rng = np.random.default_rng(seed)
    positions = seeded_population(
        n_stations, population_size, target_longitudes_rad, demands_kg, rng
    )
    velocities = np.zeros_like(positions)

    best: LayoutEvaluation | None = None

    for iteration in range(iterations):
        evaluations = [
            evaluate_layout(position, target_longitudes_rad, demands_kg, rng)
            for position in positions
        ]
        fitness = np.array([evaluation.fitness for evaluation in evaluations])

        iteration_best = evaluations[int(np.argmin(fitness))]
        if best is None or iteration_best.fitness < best.fitness:
            best = LayoutEvaluation(
                station_longitudes_rad=iteration_best.station_longitudes_rad.copy(),
                fitness=iteration_best.fitness,
                allocation=iteration_best.allocation,
            )

        best_value = fitness.min()
        worst_value = fitness.max()
        if np.isclose(best_value, worst_value):
            masses = np.full(population_size, 1.0 / population_size)
        else:
            raw_mass = np.exp(-(fitness - best_value) / (worst_value - best_value + 1e-12))
            masses = raw_mass / raw_mass.sum()

        gravitational_constant = GSA_G0 * np.exp(
            -GSA_ALPHA * iteration / max(iterations - 1, 1)
        )
        elite_count = max(
            2,
            int(round(population_size - iteration * (population_size - 2) / max(iterations - 1, 1))),
        )
        elites = np.argsort(fitness)[:elite_count]

        acceleration = np.zeros_like(positions)
        for agent in range(population_size):
            force = np.zeros(n_stations)
            for other in elites:
                if other == agent:
                    continue
                displacement = wrap_to_pi(positions[other] - positions[agent])
                distance = np.linalg.norm(displacement) + 1e-12
                force += (
                    rng.random(n_stations)
                    * gravitational_constant
                    * masses[other]
                    * displacement
                    / distance
                )
            acceleration[agent] = force

        velocities = rng.random(velocities.shape) * velocities + acceleration
        positions = wrap_to_pi(positions + velocities)

        # Elitism: never lose the global best layout.
        assert best is not None
        worst_agent = int(np.argmax(fitness))
        positions[worst_agent] = best.station_longitudes_rad.copy()
        velocities[worst_agent] = 0.0

    assert best is not None
    # Re-evaluate once so exported allocations exactly match the final centers.
    return evaluate_layout(
        best.station_longitudes_rad,
        target_longitudes_rad,
        demands_kg,
        np.random.default_rng(seed + 1_000_000),
    )


def best_of_restarts(
    n_stations: int,
    target_longitudes_rad: np.ndarray,
    demands_kg: np.ndarray,
    population_size: int,
    iterations: int,
    restarts: int,
    seed_offset: int,
) -> LayoutEvaluation:
    best: LayoutEvaluation | None = None
    for restart in range(restarts):
        candidate = gravitational_search(
            n_stations=n_stations,
            target_longitudes_rad=target_longitudes_rad,
            demands_kg=demands_kg,
            population_size=population_size,
            iterations=iterations,
            seed=RANDOM_SEED + seed_offset + 10_000 * restart,
        )
        if best is None or candidate.fitness < best.fitness:
            best = candidate
    assert best is not None
    return best


In [ ]:
# %% STEP 9 — Determine the minimum feasible number of fuel stations first


def capacity_lower_bound(demands_kg: np.ndarray) -> int:
    return max(1, int(math.ceil(demands_kg.sum() / STATION_INVENTORY_KG)))


def find_minimum_feasible_station_count(
    target_longitudes_rad: np.ndarray,
    demands_kg: np.ndarray,
) -> tuple[int, pd.DataFrame, dict[int, LayoutEvaluation]]:
    lower_bound = capacity_lower_bound(demands_kg)
    if lower_bound > MAX_STATIONS_TO_TEST:
        raise ValueError(
            f"Capacity alone requires at least {lower_bound} stations, but "
            f"MAX_STATIONS_TO_TEST={MAX_STATIONS_TO_TEST}."
        )

    records: list[dict[str, float | int | bool]] = []
    layouts: dict[int, LayoutEvaluation] = {}
    minimum_feasible: int | None = None

    for k in range(lower_bound, MAX_STATIONS_TO_TEST + 1):
        layout = best_of_restarts(
            n_stations=k,
            target_longitudes_rad=target_longitudes_rad,
            demands_kg=demands_kg,
            population_size=SCREEN_POPULATION,
            iterations=SCREEN_ITERATIONS,
            restarts=SCREEN_RESTARTS,
            seed_offset=1_000 * k,
        )
        layouts[k] = layout
        allocation = layout.allocation
        feasible = allocation.unassigned_count == 0 and np.all(
            allocation.station_load_kg <= STATION_INVENTORY_KG + 1e-8
        )
        records.append(
            {
                "STATIONS": k,
                "FEASIBLE": feasible,
                "UNASSIGNED_TARGETS": allocation.unassigned_count,
                "TOTAL_OBJECTIVE": allocation.total_objective,
                "TOTAL_SORTIE_DV_KM_S": allocation.total_sortie_dv_km_s,
                "MAX_ASSIGNED_SORTIE_DV_KM_S": float(
                    np.nanmax(allocation.assigned_sortie_dv_km_s)
                ),
                "MAX_STATION_LOAD_KG": float(allocation.station_load_kg.max()),
            }
        )
        print(
            f"k={k}: feasible={feasible}, "
            f"unassigned={allocation.unassigned_count}, "
            f"objective={allocation.total_objective:.6f}"
        )
        if feasible and minimum_feasible is None:
            minimum_feasible = k
            # Evaluate a few additional k values for a useful cost trade curve.
            if k + 3 < MAX_STATIONS_TO_TEST:
                stop_after = k + 3
            else:
                stop_after = MAX_STATIONS_TO_TEST
        if minimum_feasible is not None and k >= stop_after:
            break

    if minimum_feasible is None:
        raise RuntimeError(
            "No feasible layout was found. Increase MAX_STATIONS_TO_TEST, "
            "increase MAX_SORTIE_DV_KM_S/STATION_INVENTORY_KG, or lengthen the search."
        )

    return minimum_feasible, pd.DataFrame(records), layouts


In [ ]:
# %% STEP 10 — Re-optimize the selected k for minimum total cost


def optimize_final_layout(
    n_stations: int,
    target_longitudes_rad: np.ndarray,
    demands_kg: np.ndarray,
) -> LayoutEvaluation:
    print(f"\nFinal optimization for k={n_stations}...")
    layout = best_of_restarts(
        n_stations=n_stations,
        target_longitudes_rad=target_longitudes_rad,
        demands_kg=demands_kg,
        population_size=FINAL_POPULATION,
        iterations=FINAL_ITERATIONS,
        restarts=FINAL_RESTARTS,
        seed_offset=500_000,
    )
    if layout.allocation.unassigned_count:
        raise RuntimeError(
            "The final search lost feasibility. Increase FINAL_RESTARTS/ITERATIONS."
        )
    return layout


In [ ]:
# %% STEP 11 — Convert the optimized layout into research-friendly tables


def make_result_tables(
    targets: pd.DataFrame,
    layout: LayoutEvaluation,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    assignment = layout.allocation.assignment
    stations_rad = layout.station_longitudes_rad.copy()

    # Sort station labels west-to-east for stable, human-readable output.
    station_sort = np.argsort(stations_rad)
    old_to_new = np.empty(len(stations_rad), dtype=int)
    old_to_new[station_sort] = np.arange(len(stations_rad))
    sorted_stations = stations_rad[station_sort]
    relabeled_assignment = old_to_new[assignment]

    assignment_table = targets[
        [
            "NORAD_CAT_ID",
            "OBJECT_NAME",
            "EPOCH",
            "ANALYSIS_EPOCH_UTC",
            "LONGITUDE_DEG",
            "GEOCENTRIC_LATITUDE_DEG",
            "DEMAND_KG",
        ]
    ].copy()
    assignment_table["STATION_ID"] = relabeled_assignment + 1
    assignment_table["SORTIE_DV_KM_S"] = layout.allocation.assigned_sortie_dv_km_s
    assignment_table["OBJECTIVE_COST"] = layout.allocation.assigned_cost

    station_records: list[dict[str, float | int]] = []
    for new_index, old_index in enumerate(station_sort):
        members = assignment == old_index
        station_records.append(
            {
                "STATION_ID": new_index + 1,
                "LONGITUDE_DEG": float(np.rad2deg(sorted_stations[new_index])),
                "ASSIGNED_TARGETS": int(members.sum()),
                "ASSIGNED_DEMAND_KG": float(targets.loc[members, "DEMAND_KG"].sum()),
                "REMAINING_CAPACITY_KG": float(
                    STATION_INVENTORY_KG - targets.loc[members, "DEMAND_KG"].sum()
                ),
                "TOTAL_SORTIE_DV_KM_S": float(
                    assignment_table.loc[members, "SORTIE_DV_KM_S"].sum()
                ),
                "MAX_SORTIE_DV_KM_S": float(
                    assignment_table.loc[members, "SORTIE_DV_KM_S"].max()
                ),
            }
        )

    return pd.DataFrame(station_records), assignment_table


In [ ]:

# %% STEP 12 — Validate and export all results


def validate_final_result(
    targets: pd.DataFrame,
    stations: pd.DataFrame,
    assignments: pd.DataFrame,
) -> None:
    assert len(assignments) == len(targets)
    assert assignments["STATION_ID"].notna().all()
    assert assignments["STATION_ID"].between(1, len(stations)).all()
    assert stations["ASSIGNED_DEMAND_KG"].le(STATION_INVENTORY_KG + 1e-8).all()
    assert np.isclose(stations["ASSIGNED_DEMAND_KG"].sum(), targets["DEMAND_KG"].sum())
    if MAX_SORTIE_DV_KM_S is not None:
        assert assignments["SORTIE_DV_KM_S"].le(MAX_SORTIE_DV_KM_S + 1e-8).all()


def plot_longitude_assignments(
    assignments: pd.DataFrame,
    stations: pd.DataFrame,
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(13, 5.5))
    cmap = plt.get_cmap("tab10")

    for station_id in stations["STATION_ID"]:
        members = assignments[assignments["STATION_ID"] == station_id]
        color = cmap((station_id - 1) % 10)
        y = np.full(len(members), station_id, dtype=float)
        ax.scatter(
            members["LONGITUDE_DEG"],
            y,
            s=30 + 0.2 * members["DEMAND_KG"],
            color=color,
            alpha=0.65,
            label=f"FS-{station_id} targets",
        )
        station_lon = stations.loc[
            stations["STATION_ID"] == station_id, "LONGITUDE_DEG"
        ].iloc[0]
        ax.scatter(
            [station_lon],
            [station_id],
            marker="*",
            s=260,
            color="black",
            edgecolor="white",
            linewidth=0.8,
            zorder=5,
        )

    ax.set_xlim(-180, 180)
    ax.set_xticks(np.arange(-180, 181, 30))
    ax.set_xlabel("Earth-fixed longitude at analysis epoch (deg)")
    ax.set_ylabel("Assigned fuel station")
    ax.set_yticks(stations["STATION_ID"])
    ax.grid(alpha=0.25)
    ax.set_title("Capacity-constrained GEO fuel-station location/allocation")
    ax.legend(ncol=2, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.15))
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def plot_station_trade(trade: pd.DataFrame, output_path: Path) -> None:
    feasible = trade[trade["FEASIBLE"]].copy()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(
        feasible["STATIONS"],
        feasible["TOTAL_SORTIE_DV_KM_S"],
        marker="o",
        linewidth=2,
    )
    ax.set_xlabel("Number of fuel stations")
    ax.set_ylabel("Total round-trip sortie delta-v (km/s)")
    ax.set_title("Station-count trade study")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def export_results(
    excluded: pd.DataFrame,
    trade: pd.DataFrame,
    stations: pd.DataFrame,
    assignments: pd.DataFrame,
    analysis_epoch: pd.Timestamp,
) -> None:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    excluded.to_csv(OUTPUT_DIRECTORY / "excluded_targets.csv", index=False)
    trade.to_csv(OUTPUT_DIRECTORY / "station_count_trade.csv", index=False)
    stations.to_csv(OUTPUT_DIRECTORY / "optimal_station_locations.csv", index=False)
    assignments.to_csv(OUTPUT_DIRECTORY / "target_assignments.csv", index=False)

    plot_longitude_assignments(
        assignments, stations, OUTPUT_DIRECTORY / "longitude_assignments.png"
    )
    plot_station_trade(trade, OUTPUT_DIRECTORY / "station_count_trade.png")

    assumptions = {
        "paper_doi": "10.1016/j.asr.2020.07.028",
        "analysis_epoch_utc": analysis_epoch.isoformat(),
        "demo_mode": DEMO_MODE,
        "uniform_target_demand_kg": (
            UNIFORM_TARGET_DEMAND_KG if DEMAND_CSV is None else None
        ),
        "demand_csv": str(DEMAND_CSV) if DEMAND_CSV is not None else None,
        "station_inventory_kg": STATION_INVENTORY_KG,
        "ssc_payload_capacity_kg": SSC_PAYLOAD_CAPACITY_KG,
        "max_sortie_dv_km_s": MAX_SORTIE_DV_KM_S,
        "phasing_revolutions": PHASING_REVOLUTIONS,
        "objective_mode": OBJECTIVE_MODE,
        "ssc_dry_mass_kg": SSC_DRY_MASS_KG,
        "ssc_isp_seconds": SSC_ISP_SECONDS,
        "random_seed": RANDOM_SEED,
    }
    with (OUTPUT_DIRECTORY / "run_assumptions.json").open("w", encoding="utf-8") as handle:
        json.dump(assumptions, handle, indent=2)


In [35]:

# %% STEP 13 — Run the complete methodology in order


def main() -> None:
    if DEMO_MODE:
        print("*** DEMONSTRATION MODE: q_i and C are placeholder assumptions. ***\n")

    status, active, history = load_inputs()
    latest, excluded = latest_states_for_catalog(status, active, history)
    targets, analysis_epoch = prepare_target_locations(latest)
    targets = attach_demands(targets)

    target_longitudes = targets["LONGITUDE_RAD"].to_numpy()
    demands = targets["DEMAND_KG"].to_numpy()

    lower_bound = capacity_lower_bound(demands)
    print(f"Capacity lower bound: {lower_bound} station(s)")
    minimum_k, trade, _ = find_minimum_feasible_station_count(
        target_longitudes, demands
    )
    print(f"Minimum feasible station count found: {minimum_k}")

    final_layout = optimize_final_layout(minimum_k, target_longitudes, demands)
    stations, assignments = make_result_tables(targets, final_layout)
    validate_final_result(targets, stations, assignments)
    export_results(excluded, trade, stations, assignments, analysis_epoch)

    print("\nOptimized station table:")
    print(stations.to_string(index=False))
    print(f"\nResults written to: {OUTPUT_DIRECTORY.resolve()}")
    if DEMO_MODE:
        print(
            "\nDo not report these locations as final until DEMAND_KG, "
            "STATION_INVENTORY_KG, and the sortie constraint are replaced."
        )


if __name__ == "__main__":
    main()


*** DEMONSTRATION MODE: q_i and C are placeholder assumptions. ***

Status targets: 124
Active GPZ catalog objects: 570
History records: 12,601
History satellites: 102
Status targets also present in active GPZ catalog: 124/124
Usable targets: 102
Excluded targets: 22
Common analysis epoch: 2026-09-03T20:48:38.536416+00:00
Capacity lower bound: 5 station(s)


C:\Users\leo.langou\AppData\Local\Temp\ipykernel_1184\2140451854.py:11: UserWarning: DEMO_MODE: using a uniform placeholder demand for every target. Replace this before reporting mission-design results.
  targets = attach_demands(targets)


k=5: feasible=True, unassigned=0, objective=17.782237
k=6: feasible=True, unassigned=0, objective=14.789535
k=7: feasible=True, unassigned=0, objective=12.875278
k=8: feasible=True, unassigned=0, objective=10.656757
Minimum feasible station count found: 5

Final optimization for k=5...

Optimized station table:
 STATION_ID  LONGITUDE_DEG  ASSIGNED_TARGETS  ASSIGNED_DEMAND_KG  REMAINING_CAPACITY_KG  TOTAL_SORTIE_DV_KM_S  MAX_SORTIE_DV_KM_S
          1    -126.120085                21              2100.0                  400.0              4.116549            0.598992
          2    -103.718553                25              2500.0                    0.0              1.460866            0.121054
          3     -80.872084                21              2100.0                  400.0              3.471234            0.409594
          4      24.358035                15              1500.0                 1000.0              3.831826            0.478080
          5     105.839941           